In [1]:
# import libraries
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
import random

In [2]:
# reading data
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [3]:
# Build datasets
block_size = 3

@torch.no_grad()
def build_dataset(words):
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        
        for char in w + '.':
            ix = stoi[char]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [stoi[char]]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
with torch.no_grad():
    Xtr, Ytr = build_dataset(words[:n1])
    Xdev, Ydev = build_dataset(words[n1:n2])
    Xtest, Ytest = build_dataset(words[n2:])

In [4]:
# Initialize parameters
n_embd = 20
n_hidden = 300

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_embd), generator=g)
W1 = torch.randn((block_size*n_embd, n_hidden), generator=g) * (5/3) / ((block_size*n_embd)**0.5) # 5/3 is the specific number for tanh
#b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, 27), generator=g) * 0.1
b2 = torch.randn(27, generator=g) * 0.01

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

In [5]:
# use for finding optimal learning rate
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre
lri = []
lossi = []
stepi = []

In [104]:
# training
iterations = 100000
batch_size = 100

for i in range(iterations):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    
    emb = C[Xtr[ix]]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 #+ b1
    bnmeani = hpreact.mean(0, keepdims=True)
    bnstdi = hpreact.std(0, keepdims=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
        
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    if i % 5000 == 0:
        print(f'step {i}th: {loss.item()}')
    for p in parameters:
        p.grad = None
    loss.backward()
    
    lr = 0.1 if i <= 50000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # lri.append(lre[i])
    # lossi.append(loss.item())
    # stepi.append(i)

step 0th: 3.7154603004455566
step 5000th: 2.303460121154785
step 10000th: 2.065096378326416
step 15000th: 2.1018929481506348
step 20000th: 2.1820664405822754
step 25000th: 2.2333388328552246
step 30000th: 2.1196811199188232
step 35000th: 2.1581482887268066
step 40000th: 2.2314200401306152
step 45000th: 2.061105251312256
step 50000th: 2.236452102661133
step 55000th: 1.7389788627624512
step 60000th: 1.8214221000671387
step 65000th: 2.0013508796691895
step 70000th: 2.2062437534332275
step 75000th: 1.9286161661148071
step 80000th: 2.0864622592926025
step 85000th: 1.7840933799743652
step 90000th: 1.984768271446228
step 95000th: 1.9511610269546509


In [108]:
# Calculate loss
@torch.no_grad()
def split_loss(split):
    x, y = {'train': [Xtr, Ytr],
            'dev': [Xdev, Ydev],
            'test': [Xtest, Ytest]}[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 #+ b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(loss.item())

In [109]:
split_loss('train')
split_loss('dev')

2.0056111812591553
2.0774319171905518


In [110]:
# Get samples (names) from the machine
for _ in range(10):
    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor(context)]
        embcat = emb.view(1, -1)
        hpreact = embcat @ W1 #+ b1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))
        

eastin.
kette.
jakyit.
elke.
manity.
zequad.
mabylah.
loge.
amrey.
tri.
